In [1]:
import os
import time
import copy
import random
import numpy as np


import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import timm

import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator, MaxNLocator
from IPython.display import clear_output

import datetime
import csv


d:\Data\DL\SuitClassification\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class SimpleLogger:
    """เขียนลงไฟล์อย่างเดียว (ไม่ print)"""
    def __init__(self, log_path):
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        self.log_path = log_path
        self.f = open(log_path, "a", encoding="utf-8")
        self.file(f"\n==== RUN @ {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ====\n")

    def file(self, msg: str):
        self.f.write(msg + "\n")
        self.f.flush()

    def close(self):
        self.f.close()


class LiveHistory:
    """
    เก็บ history แล้ว:
    - เซฟเป็น metrics.csv
    - เซฟกราฟเป็น .png ทุก epoch (หรือทุกครั้งที่ plot)
    """
    def __init__(self, out_dir, title=""):
        self.out_dir = out_dir
        os.makedirs(self.out_dir, exist_ok=True)

        self.title = title
        self.rows = []  # list of dict

        self.csv_path = os.path.join(self.out_dir, "metrics.csv")
        self.png_path = os.path.join(self.out_dir, "plot.png")

        # เขียน header csv ถ้ายังไม่มี
        if not os.path.exists(self.csv_path):
            with open(self.csv_path, "w", newline="", encoding="utf-8") as f:
                w = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "val_loss", "val_acc", "val_f1"])
                w.writeheader()

    def add(self, epoch, train_loss, val_loss, val_acc, val_f1):
        row = {
            "epoch": int(epoch),
            "train_loss": float(train_loss) if train_loss is not None else None,
            "val_loss": float(val_loss) if val_loss is not None else None,
            "val_acc": float(val_acc) if val_acc is not None else None,
            "val_f1": float(val_f1) if val_f1 is not None else None,
        }
        self.rows.append(row)

        # append csv
        with open(self.csv_path, "a", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=["epoch", "train_loss", "val_loss", "val_acc", "val_f1"])
            w.writerow(row)

    def plot(self, save=True, show=True):
        if len(self.rows) == 0:
            return

        epochs = [r["epoch"] for r in self.rows]
        train_loss = [r["train_loss"] for r in self.rows]
        val_loss = [r["val_loss"] for r in self.rows]
        val_acc  = [r["val_acc"]  for r in self.rows]
        val_f1   = [r["val_f1"]   for r in self.rows]

        # ใช้โฟลเดอร์เดียวกับ self.png_path
        out_dir = os.path.dirname(self.png_path) if hasattr(self, "png_path") else getattr(self, "out_dir", ".")
        os.makedirs(out_dir, exist_ok=True)

        loss_path = os.path.join(out_dir, "plot_loss.png")
        metric_path = os.path.join(out_dir, "plot_metrics.png")

        def beautify_axis(ax):
            ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=10))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=10))
            ax.xaxis.set_minor_locator(AutoMinorLocator(2))
            ax.yaxis.set_minor_locator(AutoMinorLocator(2))
            ax.grid(True, which="major", linestyle="-", alpha=0.35)
            ax.grid(True, which="minor", linestyle=":", alpha=0.25)

        # =========================
        # (1) LOSS FIGURE
        # =========================
        fig1 = plt.figure(figsize=(10, 4.5), dpi=120)
        ax1 = fig1.gca()

        ax1.plot(epochs, train_loss, label="train_loss")
        if any(v is not None for v in val_loss):
            ax1.plot(epochs, val_loss, label="val_loss")

        ax1.set_title(f"{self.title} | Loss")
        ax1.set_xlabel("epoch")
        ax1.set_ylabel("loss")
        ax1.legend()
        beautify_axis(ax1)
        fig1.tight_layout()

        if save:
            fig1.savefig(loss_path, dpi=200, bbox_inches="tight")
        if show:
            fig1.show()
        plt.close(fig1)

        # =========================
        # (2) METRICS FIGURE
        # =========================
        fig2 = plt.figure(figsize=(10, 4.5), dpi=120)
        ax2 = fig2.gca()

        if any(v is not None for v in val_acc):
            ax2.plot(epochs, val_acc, label="val_acc")
        if any(v is not None for v in val_f1):
            ax2.plot(epochs, val_f1, label="val_f1")

        ax2.set_title(f"{self.title} | Metrics")
        ax2.set_xlabel("epoch")
        ax2.set_ylabel("score")
        ax2.set_ylim(0, 1.0)  # acc/f1 ส่วนใหญ่ 0-1 ถ้าไม่อยากล็อค ลบบรรทัดนี้ได้
        ax2.legend()
        beautify_axis(ax2)
        fig2.tight_layout()

        if save:
            fig2.savefig(metric_path, dpi=200, bbox_inches="tight")
        if show:
            fig2.show()
        plt.close(fig2)


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # ให้ผลนิ่งขึ้น (แลกกับความเร็วบางส่วน)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False


def unfreeze_last_blocks_efficientnetv2(model, n_blocks: int):
    """
    แนวคิด: EfficientNetV2 ใน timm มักมี model.blocks เป็น Sequential ของ stage/block
    เราจะ unfreeze "ท้ายสุด n_blocks" + classifier head เสมอ
    ถ้าโครงสร้างต่างออกไป จะ fallback unfreeze ทั้ง model (พร้อมเตือน)
    """
    # 1) freeze ก่อน
    freeze_all(model)

    # 2) unfreeze head/classifier เสมอ
    for name, p in model.named_parameters():
        if any(k in name.lower() for k in ["classifier", "head", "fc"]):
            p.requires_grad = True

    # 3) unfreeze blocks ท้าย ๆ
    if hasattr(model, "blocks"):
        blocks = model.blocks
        total = len(blocks)
        n = max(0, min(n_blocks, total))
        for i in range(total - n, total):
            for p in blocks[i].parameters():
                p.requires_grad = True
        return f"Unfroze last {n}/{total} blocks + head"
    else:
        # fallback (บางรุ่นอาจไม่มี .blocks)
        for p in model.parameters():
            p.requires_grad = True
        return "WARNING: model has no .blocks; unfroze ALL parameters"


In [4]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    n_total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = nn.CrossEntropyLoss()(logits, y)

        total_loss += loss.item() * x.size(0)
        n_total += x.size(0)

        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().tolist())
        all_labels.extend(y.detach().cpu().tolist())

    avg_loss = total_loss / max(1, n_total)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1

def train_one_phase(
    model, loader, device, optimizer, epochs,
    val_loader=None, history=None, phase_name="", trial_name="",
    patience=5, min_delta=0.0, verbose=True,
    print_every=50,          # ความถี่ print/log batch
    logger=None,             # logger.file(...) เก็บลงไฟล์
    save_every_epoch_plot=True
):
    criterion = nn.CrossEntropyLoss()
    best_state = copy.deepcopy(model.state_dict())
    best_val_f1 = -1.0
    bad_epochs = 0

    total_batches = len(loader)

    def log_file(msg: str):
        if logger is not None:
            logger.file(msg)

    log_file(f"== {trial_name} | {phase_name} | epochs={epochs} ==")

    for ep in range(1, epochs + 1):
        # ---- ขึ้น epoch ใหม่: ล้าง batch เก่าออกจากจอ ----
        clear_output(wait=True)
        print(f"[{trial_name}] {phase_name} | epoch {ep}/{epochs}")

        model.train()
        total_loss = 0.0
        n_total = 0
        ep_t0 = time.time()

        # ---- loop batch ----
        for b_idx, (x, y) in enumerate(loader, start=1):
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)
            n_total += x.size(0)

            # ---- print batch progress (แสดงใน epoch นี้) ----
            if (b_idx == 1) or (b_idx % print_every == 0) or (b_idx == total_batches):
                print(f"  batch {b_idx:04d}/{total_batches:04d} | loss={loss.item():.4f}")
                log_file(f"[ep {ep:02d}] batch {b_idx:04d}/{total_batches:04d} | loss={loss.item():.6f}")

        train_loss = total_loss / max(1, n_total)
        ep_sec = time.time() - ep_t0

        # ---- validation + epoch summary ----
        if val_loader is not None:
            val_loss, val_acc, val_f1 = evaluate(model, val_loader, device)

            improved = False
            if val_f1 > best_val_f1 + min_delta:
                best_val_f1 = val_f1
                best_state = copy.deepcopy(model.state_dict())
                bad_epochs = 0
                improved = True
            else:
                bad_epochs += 1

            if history is not None:
                history.title = f"{trial_name} | {phase_name}"
                history.add(ep, train_loss, val_loss, val_acc, val_f1)
                if save_every_epoch_plot:
                    history.plot(save=True, show=True)

            print(
                f"  >>> ep {ep:02d}/{epochs} DONE | train_loss={train_loss:.4f} | "
                f"val_loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f} | "
                f"bad={bad_epochs}/{patience} | {ep_sec:.1f}s" + (" ✅" if improved else "")
            )

            log_file(
                f"[ep {ep:02d}] train_loss={train_loss:.6f} val_loss={val_loss:.6f} "
                f"acc={val_acc:.6f} f1={val_f1:.6f} bad={bad_epochs}/{patience} sec={ep_sec:.2f}"
                + (" IMPROVED" if improved else "")
            )

            if bad_epochs >= patience:
                if verbose:
                    print(f"  🛑 Early stop at ep {ep} (best val_f1={best_val_f1:.4f})")
                    log_file(f"EARLY_STOP at ep={ep} best_val_f1={best_val_f1:.6f}")
                break

        

        else:
            if history is not None:
                history.title = f"{trial_name} | {phase_name}"
                history.add(ep, train_loss, None, None, None)
                if save_every_epoch_plot:
                    history.plot(save=True, show=True)

            print(f"  >>> ep {ep:02d}/{epochs} DONE | train_loss={train_loss:.4f} | {ep_sec:.1f}s")
            log_file(f"[ep {ep:02d}] train_loss={train_loss:.6f} sec={ep_sec:.2f}")

    model.load_state_dict(best_state)
    log_file(f"== DONE {trial_name} | {phase_name} | best_val_f1={best_val_f1:.6f} ==")
    return best_val_f1


In [ ]:
set_seed(42)

DATA_DIR = "data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR   = os.path.join(DATA_DIR, "val")
TEST_DIR  = os.path.join(DATA_DIR, "test")

IMG_SIZE = 224
BATCH_SIZE = 64
NUM_CLASSES = 4

# Phase epochs3
EPOCHS_A = 5   # head-only
EPOCHS_B = 20   # fine-tune

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)
import torch
print("torch:", torch.__version__)
print("torch cuda:", torch.version.cuda)
print("gpu:", torch.cuda.get_device_name(0))
print("capability:", torch.cuda.get_device_capability(0))


DEVICE = cuda
torch: 2.11.0.dev20260212+cu128
torch cuda: 12.8
gpu: NVIDIA GeForce RTX 5070 Ti Laptop GPU
capability: (12, 0)


In [6]:
to_tensor_only = transforms.ToTensor()
train_ds = datasets.ImageFolder(TRAIN_DIR, transform=to_tensor_only)
val_ds   = datasets.ImageFolder(VAL_DIR,   transform=to_tensor_only)
test_ds  = datasets.ImageFolder(TEST_DIR,  transform=to_tensor_only)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class_names = train_ds.classes
print("Classes:", class_names)
assert len(class_names) == NUM_CLASSES, f"Expected {NUM_CLASSES} classes, got {len(class_names)}"


Classes: ['club', 'diamond', 'heart', 'spade']


In [9]:
GRID = [
    # Baseline (ปลอดภัย)
    {"lr_a": 1e-3, "wd_a": 1e-4, "lr_b": 1e-4, "wd_b": 1e-4, "unfreeze_blocks": 4},

    # เพิ่ม unfreeze เพื่อใช้ข้อมูลเยอะให้คุ้ม
    {"lr_a": 1e-3, "wd_a": 1e-4, "lr_b": 1e-4, "wd_b": 1e-4, "unfreeze_blocks": 6},
    {"lr_a": 1e-3, "wd_a": 1e-4, "lr_b": 1e-4, "wd_b": 1e-4, "unfreeze_blocks": 8},

    # ปรับ LR_B ให้แรงขึ้น (บางทีได้ผลดีเมื่อข้อมูลเยอะ)
    {"lr_a": 1e-3, "wd_a": 1e-4, "lr_b": 2e-4, "wd_b": 1e-4, "unfreeze_blocks": 6},
    {"lr_a": 1e-3, "wd_a": 1e-4, "lr_b": 2e-4, "wd_b": 1e-4, "unfreeze_blocks": 8},

    # ปรับ LR_B ให้เบาลง (กันแกว่ง/กันพัง pretrained)
    {"lr_a": 1e-3, "wd_a": 1e-4, "lr_b": 5e-5, "wd_b": 1e-4, "unfreeze_blocks": 6},
    {"lr_a": 1e-3, "wd_a": 1e-4, "lr_b": 5e-5, "wd_b": 1e-4, "unfreeze_blocks": 8},

    # เพิ่ม WD_B เผื่อบางชุด overfit
    {"lr_a": 1e-3, "wd_a": 1e-4, "lr_b": 1e-4, "wd_b": 5e-4, "unfreeze_blocks": 8},
]

MODEL_NAME = "tf_efficientnetv2_s"  # จะเปลี่ยนเป็น tf_efficientnetv2_m ก็ได้ (ใหญ่ขึ้นช้าขึ้น)

best_overall = {
    "val_f1": -1.0,
    "cfg": None,
    "state": None,
}

print("\n=== GRID SEARCH START ===")
t0 = time.time()
import os, json
from datetime import datetime

RUN_DIR = os.path.join("runs_efficientnetv2", datetime.now().strftime("%Y%m%d_%H%M%S"))
os.makedirs(RUN_DIR, exist_ok=True)

for i, cfg in enumerate(GRID, start=1):
    trial_id  = f"trial_{i:02d}"
    trial_dir = os.path.join(RUN_DIR, trial_id)
    os.makedirs(trial_dir, exist_ok=True)

    # เก็บ cfg ของ trial นี้
    with open(os.path.join(trial_dir, "cfg.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2)

    # logger ของ trial นี้ (เขียนไฟล์อย่างเดียว)
    logger = SimpleLogger(os.path.join(trial_dir, "log.txt"))

    # ชื่อที่ใช้แสดงใน log/plot
    trial_name = f"Trial {i}/{len(GRID)} | cfg={cfg}"
    logger.file(f"\n--- {trial_name} ---")

    # สร้างโมเดลใหม่ทุก trial (สำคัญ!)
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)

    # ---------------- Phase A: Freeze backbone, train head only ----------------
    freeze_all(model)
    for name, p in model.named_parameters():
        if any(k in name.lower() for k in ["classifier", "head", "fc"]):
            p.requires_grad = True

    logger.file(f"  Phase A trainable params: {count_trainable_params(model)}")

    opt_a = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["lr_a"],
        weight_decay=cfg["wd_a"]
    )

    # history + outputs for Phase A
    histA = LiveHistory(out_dir=os.path.join(trial_dir, "phaseA"),
                        title=f"{trial_id} | Phase A (head-only)")

    best_f1_a = train_one_phase(
        model, train_loader, DEVICE, opt_a,
        epochs=EPOCHS_A, val_loader=val_loader,
        history=histA,
        phase_name="Phase A (head-only)",
        trial_name=trial_name,
        patience=3, min_delta=0.001,
        logger=logger,
        print_every=50,
        save_every_epoch_plot=True
    )
    logger.file(f"  Phase A best val_f1={best_f1_a:.4f}")

    # ---------------- Phase B: Unfreeze last N blocks + head, fine-tune ----------------
    msg = unfreeze_last_blocks_efficientnetv2(model, cfg["unfreeze_blocks"])
    logger.file(f"  Phase B unfreeze: {msg}")
    logger.file(f"  Phase B trainable params: {count_trainable_params(model)}")

    opt_b = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg["lr_b"],
        weight_decay=cfg["wd_b"]
    )

    # history + outputs for Phase B
    histB = LiveHistory(out_dir=os.path.join(trial_dir, "phaseB"),
                        title=f"{trial_id} | Phase B (unfreeze {cfg['unfreeze_blocks']} blocks)")

    best_f1_b = train_one_phase(
        model, train_loader, DEVICE, opt_b,
        epochs=EPOCHS_B, val_loader=val_loader,
        history=histB,
        phase_name=f"Phase B (unfreeze {cfg['unfreeze_blocks']} blocks)",
        trial_name=trial_name,
        patience=5, min_delta=0.001,
        logger=logger,
        print_every=50,
        save_every_epoch_plot=True
    )
    logger.file(f"  Phase B best val_f1={best_f1_b:.4f}")

    # Evaluate final (after best-state load) on val for record
    val_loss, val_acc, val_f1 = evaluate(model, val_loader, DEVICE)
    logger.file(f"  Final Val: loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}")

    # Update best overall
    if val_f1 > best_overall["val_f1"]:
        best_overall["val_f1"] = val_f1
        best_overall["cfg"] = cfg
        best_overall["state"] = copy.deepcopy(model.state_dict())
        logger.file("  ✅ New BEST trial!")

        # (optional) เก็บ summary ของ best trial ไว้ในไฟล์นี้ด้วย
        with open(os.path.join(RUN_DIR, "best_summary.json"), "w", encoding="utf-8") as f:
            json.dump({
                "trial_id": trial_id,
                "val_f1": float(val_f1),
                "cfg": cfg,
                "phaseA_best_f1": float(best_f1_a),
                "phaseB_best_f1": float(best_f1_b),
            }, f, indent=2)

    logger.close()




print("\n=== GRID SEARCH DONE ===")
print("Best cfg:", best_overall["cfg"])
print("Best val_f1:", best_overall["val_f1"])
print("Time (min):", (time.time() - t0) / 60.0)

# -----------------------------
# 5) Save best model
# -----------------------------
BEST_PATH = "best_efficientnetv2_ab_grid.pth"
torch.save({
    "model_name": MODEL_NAME,
    "model_state": best_overall["state"],
    "class_names": class_names,
    "img_size": IMG_SIZE,
    "best_cfg": best_overall["cfg"],
}, BEST_PATH)
print("Saved:", BEST_PATH)

[Trial 1/8 | cfg={'lr_a': 0.001, 'wd_a': 0.0001, 'lr_b': 0.0001, 'wd_b': 0.0001, 'unfreeze_blocks': 4}] Phase A (head-only) | epoch 1/5
  batch 0001/0777 | loss=7.9495


KeyboardInterrupt: 

In [7]:
TRIAL3 = {
  "lr_a": 0.001,
  "wd_a": 0.0001,
  "lr_b": 0.0001,
  "wd_b": 0.0001,
  "unfreeze_blocks": 8
}

model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)

# Phase A
freeze_all(model)
for name, p in model.named_parameters():
    if any(k in name.lower() for k in ["classifier", "head", "fc"]):
        p.requires_grad = True

opt_a = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                          lr=TRIAL3["lr_a"], weight_decay=TRIAL3["wd_a"])
_ = train_one_phase(model, train_loader, DEVICE, opt_a, epochs=EPOCHS_A, val_loader=val_loader)

# Phase B
_ = unfreeze_last_blocks_efficientnetv2(model, TRIAL3["unfreeze_blocks"])
opt_b = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                          lr=TRIAL3["lr_b"], weight_decay=TRIAL3["wd_b"])
_ = train_one_phase(model, train_loader, DEVICE, opt_b, epochs=EPOCHS_B, val_loader=val_loader)

# Save
BEST_PATH = "03_best_trial_calssification_effNetV2.pth"
torch.save({
    "trial_id": "trial_03",
    "model_name": MODEL_NAME,
    "model_state": model.state_dict(),
    "class_names": class_names,
    "img_size": IMG_SIZE,
    "best_cfg": TRIAL3,
}, BEST_PATH)

print("✅ Saved:", BEST_PATH)


NameError: name 'MODEL_NAME' is not defined

In [11]:
model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(DEVICE)
ckpt = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        logits = model(x)
        preds = logits.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(y.tolist())

print("\n=== TEST RESULTS ===")
print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Macro F1:", f1_score(all_labels, all_preds, average="macro"))
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))
print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))


=== TEST RESULTS ===
Accuracy: 0.9991951062459755
Macro F1: 0.9991952090687113

Classification Report:
              precision    recall  f1-score   support

        club       1.00      1.00      1.00      1553
     diamond       1.00      1.00      1.00      1553
       heart       1.00      1.00      1.00      1553
       spade       1.00      1.00      1.00      1553

    accuracy                           1.00      6212
   macro avg       1.00      1.00      1.00      6212
weighted avg       1.00      1.00      1.00      6212


Confusion Matrix:
[[1550    2    0    1]
 [   0 1553    0    0]
 [   0    2 1551    0]
 [   0    0    0 1553]]


In [12]:
import torch
import timm
from torchvision import transforms
from PIL import Image
import torch.nn.functional as F

BEST_PATH = "03_best_trial_calssification_effNetV2.pth"  # เปลี่ยนเป็นไฟล์ของคุณ
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ckpt = torch.load(BEST_PATH, map_location=DEVICE)

MODEL_NAME = ckpt["model_name"]
class_names = ckpt["class_names"]
IMG_SIZE = ckpt.get("img_size", 224)
NUM_CLASSES = len(class_names)

model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

# inference transform (มาตรฐาน ImageNet)
infer_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225))
])

print("Loaded:", MODEL_NAME, "| classes:", class_names)

Loaded: tf_efficientnetv2_s | classes: ['club', 'diamond', 'heart', 'spade']


In [13]:
def predict_image(path, topk=3):
    img = Image.open(path).convert("RGB")
    x = infer_tfm(img).unsqueeze(0).to(DEVICE)  # (1,3,H,W)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1).squeeze(0)  # (C,)

    topk = min(topk, probs.numel())
    vals, idxs = torch.topk(probs, k=topk)

    results = [(class_names[i], float(v)) for v, i in zip(vals.cpu(), idxs.cpu())]
    return results

# ใช้งาน
img_path = "sample.png"   # <-- เปลี่ยนเป็นรูปของคุณ
print(predict_image(img_path, topk=4))


[('diamond', 1.0), ('spade', 0.0), ('club', 0.0), ('heart', 0.0)]


In [14]:
from torchvision import transforms

IMG_SIZE = 224

to_tensor_only = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),      # ไม่มี Normalize
])


In [16]:
from PIL import Image
import torch.nn.functional as F

infer_tfm = to_tensor_only  # ให้ตรงกับ test_ds

def predict_image(path, topk=4):
    img = Image.open(path).convert("RGB")
    x = infer_tfm(img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1).squeeze(0)

    vals, idxs = torch.topk(probs, k=min(topk, probs.numel()))
    return [(class_names[i], float(v)) for v, i in zip(vals.cpu(), idxs.cpu())]

print(predict_image("sample.png", topk=4))


[('club', 1.0), ('spade', 2.2463650880297052e-14), ('heart', 1.2246362646733272e-16), ('diamond', 3.434955505643623e-17)]
